In [1]:
import pandas as pd
import numpy as np
import pickle
import ast
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load model

In [2]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


### Load embeddings

In [3]:
with open("recipes_embeddings_list.pkl", "rb") as f:
    recipes_embeddings_list = pickle.load(f)
print(f"Loaded {len(recipes_embeddings_list)} recipe embeddings")

Loaded 10263 recipe embeddings


### Load dataset

In [4]:
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


### 3. Late Fusion Search Function

**Late Fusion Strategy:**
1. Tính similarity của query với **TẤT CẢ** các câu trong mỗi món
2. Lấy **trung bình** (average) similarity của tất cả câu → điểm số của món
3. Rank tất cả món theo điểm trung bình → Top K

In [5]:
def search_recipes_late_fusion(query, model, recipes_embeddings_list, all_recipes_df, top_k=10):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)

    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank

    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per dish
        all_recipes_df: Recipe metadata dataframe
        top_k: Number of results to return

    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize

    # 2. Calculate average similarity for EACH recipe
    recipe_scores = []

    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        if len(dish_embeds) == 0:
            continue

        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)

        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()

        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)

        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities)),
            'num_sentences': len(similarities)
        })

    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]

    # 4. Create results dataframe with FULL recipe info
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = all_recipes_df.iloc[recipe_idx]

        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'num_of_people': recipe['num_of_people'],
            'ingredients': recipe['ingredients'],
            'step': recipe['step'],
            'note': recipe['note'],
            'description': recipe['description'],
            'link': recipe['link']  # Thêm link
        })

    return pd.DataFrame(results)

### 4. Display results function

In [6]:
def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []

In [7]:
import re

def display_results(results_df, query):
    """
    Display search results với TẤT CẢ thông tin món ăn

    Args:
        results_df: DataFrame from search_recipes_late_fusion
        query: Original query string
    """
    print(f"Query: '{query}'")
    print(f"Top {len(results_df)} Results:")

    for idx, row in results_df.iterrows():
        print(f"\n{'='*100}")
        print(f"{idx+1}. [{row['avg_similarity']:.4f}] {row['title']}")
        print(f"{'='*100}")

        # Basic info
        print(f"\nTHÔNG TIN CƠ BẢN:")
        print(f"   • Loại món: {row['type_of_food']}")
        print(f"   • Thời gian nấu: {row['cook_time']}")
        print(f"   • Số người ăn: {row['num_of_people']}")
        
        # Link
        if pd.notna(row['link']):
            print(f"   • Link: {row['link']}")

        # Similarity scores
        print(f"\nĐIỂM SIMILARITY:")
        print(f"   • Trung bình (AVG): {row['avg_similarity']:.4f}")
        print(f"   • Cao nhất (MAX): {row['max_similarity']:.4f}")
        print(f"   • Thấp nhất (MIN): {row['min_similarity']:.4f}")
        print(f"   • Số câu đánh giá: {row['num_sentences']}")

        # Description
        if pd.notna(row['description']):
            print(f"\nMÔ TẢ:")
            print(f"   {row['description']}")

        # Ingredients
        ingredients = parse_list_field(row['ingredients'])
        if ingredients:
            print(f"\nNGUYÊN LIỆU ({len(ingredients)} món):")
            for i, ing in enumerate(ingredients, 1):
                print(f"   {i}. {ing}")

        # Steps
        steps = parse_list_field(row['step'])
        if steps:
            # 1. Gộp tất cả step thành 1 chuỗi
            steps_text = " ".join(step.strip() for step in steps)

            # 2. Format: gặp "Bước X:" thì xuống dòng
            steps_text = re.sub(r'(Bước\s+\d+:)', r'\n\1', steps_text).strip()

            print(f"\nCÁCH LÀM:")
            print(steps_text)

        # Notes
        notes = parse_list_field(row['note'])
        if notes:
            print(f"\nLƯU Ý:")
            for i, note in enumerate(notes, 1):
                print(f"   • {note}")

        print()

In [8]:
# Test Late Fusion
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
]

# Run Late Fusion tests
for query in test_queries:
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=5
    )

    # Display results
    display_results(results, query)

Query: 'Món ăn có thịt bò nấu nhanh'
Top 5 Results:

1. [0.6043] Bò hầm cà rốt

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 30phút
   • Số người ăn: 2
   • Link: https://vncooking.com/cong-thuc/bo-ham-ca-rot-14

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6043
   • Cao nhất (MAX): 0.7382
   • Thấp nhất (MIN): 0.4551
   • Số câu đánh giá: 4

MÔ TẢ:
   Bò luôn là món thịt mà đa phần các gia đình Việt rất ưa chuộng, bò chứa hàm lượng dinh dưỡng siêu cao cộng với cà rốt nữa làm tăng thêm phần dinh dưỡng của món ăn. Cùng chuẩn bị nguyên liệu thực hiện món Bò hầm cà rốt này nhé.

NGUYÊN LIỆU (7 món):
   1. Thịt bò 150 gram
   2. Cà rốt 2 củ
   3. Nước mắm 1 muỗng cafe
   4. Gừng 1 củ
   5. Muối 1 muỗng
   6. Dầu ăn 2 muỗng
   7. Sả 1 cây

CÁCH LÀM:
Bước 1: Nguyên liệu rửa sạch. Cà rốt cạo vỏ thái thành những hình vuông nhỏ, vừa ăn. Thịt bò cũng vậy, thái thành từng miếng hình vuống nhỏ thôi cho không bị day nha. Các nguyên liệu khác bỏ vỏ đun trên 1 nồi nước nhỏ, chờ khi nướ

### 5. Interactive Search

In [9]:
# Interactive search với Late Fusion
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("Query: ").strip()

    if query.lower() in ['quit', 'exit', 'q']:
        break

    if not query:
        continue

    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=10
    )

    # Display results
    display_results(results, query)

Enter your query (type 'quit' to exit):

Query: 'thịt kho trứng'
Top 10 Results:

1. [0.5765] Thịt bò xào trứng

THÔNG TIN CƠ BẢN:
   • Loại món: Món khai vị
   • Thời gian nấu: 45phút
   • Số người ăn: 2
   • Link: https://vncooking.com/cong-thuc/thit-bo-xao-trung-340

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.5765
   • Cao nhất (MAX): 0.6575
   • Thấp nhất (MIN): 0.4847
   • Số câu đánh giá: 4

MÔ TẢ:
   Món trứng xào thịt bò màu sắc hấp dẫn, thịt bò chín tới không quá dai

NGUYÊN LIỆU (5 món):
   1. Thịt bò 200 gram
   2. Trứng gà 3 quả
   3. Hành lá 1 nhánh
   4. Nước dùng 300 ml
   5. Rượu trắng 1 muỗng cà phê

CÁCH LÀM:
Bước 1: Thịt bò cắt lát mỏng, đem trộn đều với 1/2 muỗng cà phê rượu, 1 muỗng cà phê nước mắm, 1/2 muỗng cà phê đường, 1 muỗng canh bột bắp, sau đó thêm 1 muỗng cà phê dầu mè, đảo đều. Thịt ướp chừng 30 phút cho thấm. 
Bước 2: Trứng đập ra chén, đánh tan cùng với một muỗng cà phê nước, 1/2 muỗng cà phê rượu và 1 muỗng cà phê muối. Làm nóng 1 muỗng canh dầu trong c

In [10]:
results.head()

,recipe_idx,avg_similarity,max_similarity,min_similarity,num_sentences,title,type_of_food,cook_time,num_of_people,ingredients,step,note,description,link
0,363,0.302514,0.457622,0.180007,4,Sườn hầm ngô non,Món ngon hàng ngày,NaN,NaN,"['Sườn thăn', 'Ngô ngọt: một bắp', 'Cà rốt: mộ...","['Bước 1: Cà rốt cạo vỏ, rửa sạch cắt miếng nh...",[],"Thời tiết hanh nắng dễ mệt mỏi, bát canh ngô s...",https://vnexpress.net/suon-ham-ngo-non-4311028...
1,706,0.271479,0.363955,0.128130,3,Hầm bí đao cả quả với sườn non,Thực đơn cho ngày nắng nóng,NaN,NaN,"['Bí đao', 'Sườn lợn non']","['Bước 1: Sườn non rửa sạch, chiên tới khi chá...",[],Canh sườn non được nấu trong nguyên trái bí đa...,https://vnexpress.net/ham-bi-dao-ca-qua-voi-su...
2,347,0.253639,0.416792,0.107002,5,Món sườn non rang muối,Món ngon hàng ngày,NaN,NaN,"['500 g sườn non', '3 củ sả', 'Bột muối loại d...","['Bước 1: Sườn chặt miếng vừa ăn, rửa sạch, ch...",[],"Đã chán sườn xào chua ngọt, sườn rim mặn thì b...",https://vnexpress.net/cach-lam-mon-suon-non-ra...
3,6722,0.242675,0.352363,0.119967,7,Cách ướp sườn non chiên giòn đậm đà làm mồi nh...,Món chiên,50 phút,970gr sườn non,"['970 gr Sườn non', '1 ít Gia vị thông dụng (t...","['Bước 1: Sơ chế sườn non: Đầu tiên, chúng ta ...","['Để món sườn giòn thêm hoàn hảo, việc lựa chọ...",Bạn đang tìm cách ướp sườn non chiên giòn ngon...,https://www.dienmayxanh.com/vao-bep/cach-uop-s...
4,352,0.242529,0.295419,0.173736,4,Lạ miệng với sườn hấp bia,Món ngon hàng ngày,NaN,NaN,"['400 g sườn non', '200 g khoai tây bi', '1/2 ...","['Bước 1: Sườn lợn rửa sạch, chặt khúc vừa miệ...",[],Sườn chua ngọt hay canh sườn đã quá quen thuộc...,https://vnexpress.net/la-mieng-voi-suon-hap-bi...


### 6. Combine with state.json

#### 6.1 Load state.json

In [15]:
from pathlib import Path
import json

STATE_FILE = "state.json"

def load_state_json(filepath=STATE_FILE):
    """Load dialogue state from JSON file"""
    try:
        if Path(filepath).exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                state = json.load(f)
            return state
        else:
            print(f"[Error]: {filepath} not found")
            return None
    except json.JSONDecodeError:
        print(f"[Error]: Invalid JSON in {filepath}")
        return None

In [16]:
state = load_state_json()
state

{'hard_constraints': {'type_of_food': ['món kho'],
  'ingredients': ['sườn non']},
 'soft_constraints': {'cook_time': [],
  'num_of_people': [],
  'calories': [],
  'algeric': []},
 'recommended_items': [],
 'accepted_items': [],
 'rejected_items': []}

#### 6.2 Generate query from state (Rule-based)

In [17]:
def generate_query_from_state(state):
    parts = []
    
    # Hard constraints (priority)
    if "hard_constraints" in state:
        # Type of food
        if state["hard_constraints"].get("type_of_food"):
            type_food = state["hard_constraints"]["type_of_food"]
            if type_food and type_food[0] != "none":
                parts.append(type_food[0])
        
        # Ingredients
        if state["hard_constraints"].get("ingredients"):
            ingredients = state["hard_constraints"]["ingredients"]
            if ingredients and ingredients != ["none"]:
                if len(ingredients) == 1:
                    parts.append(f"có {ingredients[0]}")
                else:
                    parts.append(f"có {', '.join(ingredients)}")
    
    # Soft constraints
    if "soft_constraints" in state:
        # Number of people
        if state["soft_constraints"].get("num_of_people"):
            num_people = state["soft_constraints"]["num_of_people"]
            if num_people and num_people[0] != "none":
                parts.append(f"cho {num_people[0]} người")
        
        # Cook time
        if state["soft_constraints"].get("cook_time"):
            cook_time = state["soft_constraints"]["cook_time"]
            if cook_time and cook_time[0] != "none":
                parts.append(f",có thời gian nấu {cook_time[0]} phút")
    
    # Combine parts into natural query
    if not parts:
        return "Món ăn"
    
    query = " ".join(parts)
    return query

In [18]:
# Test query generation
test_query = generate_query_from_state(state)
print(f"Generated query: '{test_query}'")

Generated query: 'món kho có sườn non'


#### 6.3 Search from state.json

In [19]:
def search_from_state_json(state_filepath=STATE_FILE, top_k=10):
    """
    Args:
        state_filepath: Path to state.json file
        top_k: Number of results to return
    
    Returns:
        DataFrame with search results
    """
    # STEP 1: Load state.json
    print("\nSTEP 1: Loading state.json...")
    print("-"*100)
    state = load_state_json(state_filepath)
    
    if state is None:
        return None
    
    # STEP 2: Generate query
    print("\nSTEP 2: Generating query from constraints...")
    
    
    query = generate_query_from_state(state)
    print(f"Generated Query: '{query}'")
    print("-"*100)
    
    # STEP 3: Search recipes using Late Fusion
    print("\nSTEP 3: Searching recipes with Late Fusion...")
    
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=top_k
    )
    
    print(f"Found {len(results)} recipes")
    print("-"*100)

    # STEP 4: Display results
    print("STEP 4: RESULTS")
    display_results(results, query)
    
    return results

In [20]:
# Run search from state.json
results = search_from_state_json(
    state_filepath="state.json",
    top_k=10
)


STEP 1: Loading state.json...
----------------------------------------------------------------------------------------------------

STEP 2: Generating query from constraints...
Generated Query: 'món kho có sườn non'
----------------------------------------------------------------------------------------------------

STEP 3: Searching recipes with Late Fusion...
Found 10 recipes
----------------------------------------------------------------------------------------------------
STEP 4: RESULTS
Query: 'món kho có sườn non'
Top 10 Results:

1. [0.6783] Sườn rim chua ngọt thưởng thức cùng cơm trắng

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 30phút
   • Số người ăn: 3
   • Link: https://vncooking.com/cong-thuc/suon-rim-chua-ngot-thuong-thuc-cung-com-trang-75

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6783
   • Cao nhất (MAX): 0.7472
   • Thấp nhất (MIN): 0.5579
   • Số câu đánh giá: 3

MÔ TẢ:
   Những miếng sườn ram mềm, đậm đà vị mặn ngọt ăn kèm với cơm trắng là đúng 